# DT401 Workshop 1 — SQL Part 1: Selecting Data

## Learning Objectives

By the end of this notebook, you will be able to:
- Write a `SELECT` statement to retrieve columns from a database table
- Use `LIMIT` to control how many rows are returned
- Use `COUNT` to count records in a table
- Use `DISTINCT` and `COUNT(DISTINCT)` to identify and count unique values

---

## Context

This notebook introduces SQL — the language used to communicate with relational databases. You will be working with the **pagila database**, a sample database modelling a fictional DVD rental business. The skills you practise here form the foundation for everything that follows in this workshop and in later modules.

**This notebook has 17 questions.** Work through each question in order. Write your answer in the code cell provided, then check the solution dropdown if you need guidance. Every solution includes a brief explanation — read it even if you got the answer right.

## What is SQL?

We communicate with databases using **SQL (Structured Query Language)**, which allows us to read, write, create, and delete parts of a database. SQL is designed for a specific purpose — querying databases — which keeps it relatively straightforward to learn.

More specifically, SQL is:

- *Structured*: Writing an SQL query is like writing a sentence — the words have specific meanings and the order matters.
- *Query*: SQL is a language for asking questions of a database. You can ask it to extract data, update data, delete data, or insert data.
- *Language*: SQL has special reserved words called **keywords**, and unlike Python, it is not a general-purpose programming language.

Even if your organisation uses a visual interface for data — a dashboard, a BI tool, a CRM system — SQL is likely running behind the scenes. Understanding it gives you direct access to your data, without waiting for a report to be built for you.

<div class="alert alert-warning">

**⚠ Warning: SQL comes in several flavours**

The commands in this workshop follow the SQL standard and work in most databases. However, different database systems — [PostgreSQL](https://www.postgresql.org/), [MySQL](https://www.mysql.com/), [SQLite](https://www.sqlite.org/), and [Oracle](https://www.oracle.com/database/) — each have small differences. The pagila database uses **SQLite**. Always check the documentation for your specific database when you encounter an unexpected result.

</div>

<div class="alert alert-success">

**Further resources**

If you want additional practice or reference material:
- [SQLZoo](https://sqlzoo.net/) — Interactive SQL tutorials you can run in the browser
- [W3Schools SQL Tutorial](https://www.w3schools.com/sql/) — Comprehensive keyword reference
- [Kaggle Intro to SQL](https://www.kaggle.com/learn/intro-to-sql) — Practical examples with real datasets

</div>

## The pagila database

The [**pagila database**](https://www.postgresqltutorial.com/postgresql-sample-database/) models a fictional DVD rental business — think Blockbuster, but in a database. It contains tables for customers, films, payments, rentals, staff, stores, and more. Using a single, consistent database throughout this workshop means you can focus on learning SQL rather than switching between different data contexts.

Relational databases serving healthcare, retail, government, and finance organisations all follow the same structural principles as pagila. As you explore its tables, consider what the equivalent structures might look like in your own organisation — a `customer` table might become a `patient` table, a `staff` table might become an `employee` table, but the relationships between them work the same way.

See the [ERD notebook](ERD.ipynb) for a complete diagram of all 14 tables. Throughout this workshop series, smaller diagrams will show which tables are relevant for each topic.

The three tables used most in this notebook are:

```mermaid
erDiagram
    customer {
        integer customer_id PK
        smallint store_id
        varchar first_name
        varchar last_name
        varchar email
        smallint address_id
        boolean activebool
        date create_date
        integer active
    }
    payment {
        integer payment_id PK
        smallint customer_id
        smallint staff_id
        integer rental_id
        numeric amount
        timestamp payment_date
    }
    film {
        integer film_id PK
        varchar title
        text description
        year release_year
        smallint rental_duration
        numeric rental_rate
        smallint length
        mpaa_rating rating
    }
```

## Connecting to the database

The pagila database is a **SQLite** database — a lightweight, file-based database that does not need a separate server. It is widely used in mobile apps, embedded systems, and small-scale applications.

To run SQL inside a Jupyter notebook, we use the [`jupysql`](https://jupysql.ploomber.io/) extension. Run the setup cell below before doing anything else. If your kernel restarts at any point, run this cell again before running any queries.

In [ ]:
# Setup: load the SQL extension and connect to the pagila database
%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False
%config SqlMagic.autopolars = False
%config SqlMagic.displaycon = False

%sql sqlite:///data/pagila.db

To run SQL in a code cell, start the cell with `%%sql`. Everything after that line is treated as SQL:

```
%%sql
SELECT ...;
```

You can also run a single SQL statement inline within a Python line using `%sql`:

```python
result = %sql SELECT * FROM customer LIMIT 5;
```

Run the cell below to confirm your connection is working — it should return `2`:

In [ ]:
%%sql
SELECT 1 + 1;

## Section 1: Extracting data from a table

### Basic SELECT queries

All SQL queries that extract data start with `SELECT`. The basic structure is:

```sql
SELECT column1, column2
FROM table_name;
```

To return all columns, use `*` — a wildcard meaning "everything":

```sql
SELECT *
FROM table_name;
```

<div class="alert alert-info">

**Convention:** SQL keywords are typically written in **UPPERCASE** (e.g. `SELECT`, `FROM`). SQL is not case-sensitive, but uppercase keywords make queries easier to read at a glance.

</div>

SQL can also perform calculations directly:

```sql
SELECT 1 + 1, 2 * 2, 3 - 2;
```

<div class="alert alert-info">

**Integer division:** `3 / 2` returns `1`, not `1.5`. To get a decimal result, use a decimal point: `3.0 / 2` returns `1.5`.

</div>

### Question 01

> Use `SELECT` to extract *all* columns from the `customer` table.
> - What columns does it have?
> - What data types do you think they contain? Check the ERD diagram above to confirm.

In [ ]:
%%sql


<details>
<summary>Solution to Question 01</summary>

```sql
%%sql
SELECT *
FROM customer;
```

The table has several columns: `customer_id` (integer), `first_name` and `last_name` (text), `email` (text), `store_id` (small integer indicating which store the customer belongs to), `address_id` (integer linking to the `address` table), `activebool` (boolean — whether the customer is currently active), and `create_date` (the date the customer record was created). Compare what you see with the ERD diagram above to check your data type guesses.

</details>

### Question 02

> Extract only the `first_name` and `last_name` columns from the `customer` table.
> - How many records are returned?

In [ ]:
%%sql


<details>
<summary>Solution to Question 02</summary>

```sql
%%sql
SELECT first_name, last_name
FROM customer;
```

599 records are returned — there are 599 customers in the database. When you select specific columns rather than `*`, the result contains the same number of rows but only the columns you asked for.

</details>

### 1.2 Limiting the number of records

Large tables can return thousands of rows. Use `LIMIT` to restrict how many records are returned:

```sql
SELECT *
FROM table_name
LIMIT 10;
```

This is especially useful when you first explore a new table — you want to see its structure without waiting for the entire dataset to load.

### Question 03

> Use `LIMIT` to return only the first 10 records from `customer`.

In [ ]:
%%sql


<details>
<summary>Solution to Question 03</summary>

```sql
%%sql
SELECT *
FROM customer
LIMIT 10;
```

`LIMIT` is especially useful when exploring large tables — it lets you inspect the structure and data types without loading thousands of rows. It is good practice to use `LIMIT` any time you run `SELECT *` on a table you haven't examined before.

</details>

### 1.3 Counting records with `COUNT`

**Functions** take an input, perform an operation, and return a result. `COUNT` counts the number of records returned by a query:

```sql
SELECT COUNT(*)
FROM actor;
```

### Question 04

> Use `COUNT` to count all records in the `customer` table. How many are there?

In [ ]:
%%sql


<details>
<summary>Solution to Question 04</summary>

```sql
%%sql
SELECT COUNT(*)
FROM customer;
```

599 records. This matches the row count from Question 02 — a useful cross-check. `COUNT(*)` counts every row in the table, including any with NULL values in individual columns.

</details>

### Question 05

> Explore the `payment` table:
> - Extract all columns (use `LIMIT 10` to avoid loading everything)
> - What are the column names and data types?
> - How many records are there in total?

In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution to Question 05</summary>

```sql
%%sql
SELECT *
FROM payment
LIMIT 10;
```

```sql
%%sql
SELECT COUNT(*)
FROM payment;
```

There are 16,049 records in the `payment` table — significantly more than the 599 customers, because most customers have made multiple payments. The columns are: `payment_id` (integer, unique identifier for each payment), `customer_id` (integer, linking to the customer who made the payment), `staff_id` (integer, the staff member who processed it), `rental_id` (integer, the rental being paid for), `amount` (numeric, the payment value), and `payment_date` (timestamp).

</details>

### Question 06

> Extract the columns `customer_id`, `amount`, and `payment_date` from the `payment` table.
> - What is the data type of each column?

In [ ]:
%%sql


<details>
<summary>Solution to Question 06</summary>

```sql
%%sql
SELECT customer_id, amount, payment_date
FROM payment
LIMIT 10;
```

- `customer_id`: integer — a whole number identifying the customer
- `amount`: numeric — a decimal number representing the payment value in dollars
- `payment_date`: timestamp — a date and time value recording when the payment was made

Adding `LIMIT 10` here is good practice — it lets you confirm the column contents without loading all 16,049 rows.

</details>

### Question 07

> Choose any table you haven't looked at yet. Extract all of its data.
> - What are its columns?
> - What data types do they contain?
> - How many records does it have?

In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Example Solution — Question 07</summary>

```sql
%%sql
SELECT *
FROM film
LIMIT 10;
```

```sql
%%sql
SELECT COUNT(*)
FROM film;
```

The `film` table has 1,000 records. Its columns include `film_id`, `title`, `description`, `release_year`, `rental_duration`, `rental_rate`, `length`, and `rating`. Try a different table — `actor`, `store`, `staff`, `language`, or `category` are all worth exploring. Notice how each table has its own set of columns suited to the kind of information it holds.

</details>

### Question 08

> Use `SELECT` to view columns in both the `city` and `country` tables. You will need two queries.
> - Do `city` and `country` share any column names?
>
> **Challenge:** Find two other tables that share a column name.

In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution to Question 08</summary>

```sql
%%sql
SELECT *
FROM city
LIMIT 5;
```

```sql
%%sql
SELECT *
FROM country
LIMIT 5;
```

Both `city` and `country` have a `country_id` column. Similarly, both `rental` and `payment` share the `rental_id` column.

Shared column names often indicate a relationship between tables — the `country_id` in `city` is a reference back to a row in `country`. You will explore these relationships in depth in the JOINs notebook.

</details>

### Question 09

> Extract just the `first_name` column from the `customer` table. Are all the names unique?
>
> *Hint: compare the total row count with what you know from Question 02.*

In [ ]:
%%sql


<details>
<summary>Solution to Question 09</summary>

```sql
%%sql
SELECT first_name
FROM customer;
```

The query returns 599 rows — the same as the total customer count. But if you scan the results, you will notice some names appear more than once. "Mary", "Linda", and "Patricia" each appear several times. The list contains duplicates because multiple customers can share a first name. In the next section, you will learn how to extract only unique values.

</details>

### 1.4 Extracting unique values with `DISTINCT`

Many columns contain repeated values. Use `DISTINCT` to see only the unique values present:

```sql
SELECT DISTINCT(column_name)
FROM table_name;
```

You can combine `DISTINCT` with `COUNT` to count the number of unique values:

```sql
SELECT COUNT(DISTINCT(column_name))
FROM table_name;
```

### Question 10

> Use `DISTINCT` to extract all the distinct first names from `customer`.

In [ ]:
%%sql


<details>
<summary>Solution to Question 10</summary>

```sql
%%sql
SELECT DISTINCT(first_name)
FROM customer;
```

This returns fewer rows than a plain `SELECT first_name` query because duplicate names are collapsed into a single entry. You can see at a glance which first names are used in the database without sifting through 599 rows of repeated values.

</details>

### Question 11

> How many *distinct* first names are there in `customer`?
>
> <details>
> <summary>💡 Hint</summary>
>
> `SELECT COUNT(first_name) FROM customer;` does **not** give the right answer — it counts all rows, including duplicates. You need to count only the unique ones.
>
> </details>

In [ ]:
%%sql


<details>
<summary>Solution to Question 11</summary>

```sql
%%sql
SELECT COUNT(DISTINCT(first_name))
FROM customer;
```

591 distinct first names out of 599 customers — so 8 first names are shared by more than one customer. `COUNT(DISTINCT(column))` is a common pattern when you want to know how many different values exist in a column, rather than how many rows contain a value.

</details>

### Question 12

> How many distinct values for `amount` exist in the `payment` table?

In [ ]:
%%sql


<details>
<summary>Solution to Question 12</summary>

```sql
%%sql
SELECT COUNT(DISTINCT(amount))
FROM payment;
```

19 distinct values for `amount`. Despite 16,049 payment records, customers are only ever charged one of 19 possible amounts — this reflects the fixed rental rates for different film categories in the pagila business model. If you want to see what those 19 amounts are, try `SELECT DISTINCT(amount) FROM payment;`.

</details>

### Question 13

> How many different `store_id` values exist in the `customer` table?

In [ ]:
%%sql


<details>
<summary>Solution to Question 13</summary>

```sql
%%sql
SELECT COUNT(DISTINCT(store_id))
FROM customer;
```

2 distinct values for `store_id` — each customer belongs to one of two stores. This makes sense for a small DVD rental chain. In the next question, you will verify this against the `store` table itself.

</details>

### Question 14

> - How many records are there in the `store` table?
> - Do all possible `store_id` values from the `store` table appear in the `customer` table?

In [ ]:
%%sql


<details>
<summary>Solution to Question 14</summary>

```sql
%%sql
SELECT COUNT(*)
FROM store;
```

There are 2 records in the `store` table, which matches the 2 distinct `store_id` values you found in `customer`. This confirms that every store has at least one customer — both `store_id` values in `store` appear in `customer`. Checking consistency across tables like this is a good habit when exploring an unfamiliar database.

</details>

### Question 15

> - How many different cities are there in the `address` table (measured by distinct `city_id` values)?
> - How many cities are there in the `city` table?
> - Is the count the same? What might explain any difference?

In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution to Question 15</summary>

```sql
%%sql
SELECT COUNT(DISTINCT(city_id))
FROM address;
```

```sql
%%sql
SELECT COUNT(*)
FROM city;
```

There are 599 distinct `city_id` values in `address`, but 600 cities in the `city` table — one city exists in the `city` table but has no address linked to it. This is a common pattern in relational databases: a reference table (like `city`) may contain entries that are not yet used anywhere. You will see this kind of relationship more clearly when you work with JOINs.

</details>

### Question 16

> How many addresses in the `address` table share a `city_id` with at least one other address?
>
> *Hint: compare the number of distinct `city_id` values with the total number of rows in `address`.*

In [ ]:
%%sql


In [ ]:
%%sql


<details>
<summary>Solution to Question 16</summary>

```sql
%%sql
SELECT COUNT(DISTINCT(city_id))
FROM address;
```

```sql
%%sql
SELECT COUNT(*)
FROM address;
```

There are 599 distinct `city_id` values but 603 total rows in `address` — so at least 4 addresses share a `city_id` with another address (603 − 599 = 4 "extra" rows beyond what would exist if every city appeared exactly once). In a real dataset this could indicate multiple branches, offices, or properties located in the same city. You will learn how to identify exactly which cities appear more than once when you reach the GROUP BY and HAVING topics in Notebook 3.

</details>

---

## Summary

In this notebook you have covered:

| Keyword / Function | Purpose |
|---|---|
| `SELECT` | Specify which columns to return |
| `FROM` | Specify which table to query |
| `*` | Wildcard — select all columns |
| `LIMIT n` | Return only the first `n` records |
| `COUNT(*)` | Count the number of records |
| `DISTINCT(col)` | Return only unique values |
| `COUNT(DISTINCT(col))` | Count unique values |

**Next:** [Notebook 2 — Filtering Data](2_filtering_data.ipynb)